# Phase 5 - Terrain and LiDAR preparation

This notebook crops the Phase 4 elevation data to the 500 m by 500 m UQ pilot, preserves an audit LAZ, creates filtered modelling and display LAZ files, derives one-metre terrain/surface rasters, and exposes the validation evidence. Outputs retain EPSG:7856 horizontal coordinates and AHD elevations.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if not (project_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the project root containing pyproject.toml')

src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pandas as pd
from IPython.display import JSON, Image, display

from uqgems.phase5 import run_phase5

project_root

## Run and validate

The initial run streams the full-resolution data. Later runs reuse products only when all source, configuration and output hashes match the manifest.

In [ ]:
summary = run_phase5(project_root)
print(f"Phase 5 status: {summary['status']}")
print(f"Cached data products reused: {summary['cache_reused']}")
display(JSON(summary['checks'], expanded=True))

## Point-cloud and quality summary

The full crop is the audit record, the analysis crop is the modelling input, and the deterministic sample is for visualisation only.

In [ ]:
stats = summary['statistics']
overview = pd.Series({
    'full crop points': summary['point_clouds']['full_laz']['point_count'],
    'analysis points': summary['point_clouds']['analysis_laz']['point_count'],
    'display points': summary['point_clouds']['display_laz']['point_count'],
    'points removed': stats['filtering']['removed_total'],
    'mean points per m2': stats['analysis']['density']['mean_points_per_m2'],
    'non-empty cell fraction': stats['analysis']['density']['nonempty_cell_fraction'],
    'ground-DTM median residual (m)': stats['analysis']['ground_dtm_comparison']['median_m'],
    'ground-DTM RMSE (m)': stats['analysis']['ground_dtm_comparison']['rmse_m'],
}, name='value')
display(overview.to_frame())

filter_rows = {
    'ASPRS classes 7/18': stats['filtering']['removed_noise_class'],
    'below -2 m relative to DTM': stats['filtering']['removed_below_height_threshold'],
    'above +80 m relative to DTM': stats['filtering']['removed_above_height_threshold'],
    'without valid DTM cell': stats['filtering']['removed_outside_valid_dtm'],
}
display(pd.Series(filter_rows, name='points removed').to_frame())

## Classification inventory

Counts are obtained by chunked scans. Zero retained points for the declared noise classes is an automated quality gate.

In [ ]:
full_classes = stats['full_classification_counts']
analysis_classes = stats['analysis']['classification_counts']
class_codes = sorted(
    {int(code) for code in full_classes} | {int(code) for code in analysis_classes}
)
class_table = pd.DataFrame([
    {
        'class': code,
        'name': (full_classes.get(str(code)) or analysis_classes[str(code)])['name'],
        'full crop': full_classes.get(str(code), {}).get('count', 0),
        'analysis crop': analysis_classes.get(str(code), {}).get('count', 0),
    }
    for code in class_codes
])
display(class_table)

## Fixed visual checks

The height map is a normalised surface raster, not yet an LoD building model. Height and density colours stop at their 99th percentiles so isolated extreme cells do not hide useful structure.

In [ ]:
for name in ['hillshade', 'height', 'density', 'classifications', 'ground_residual']:
    print(name.replace('_', ' ').title())
    display(Image(filename=str(project_root / summary['outputs']['figures'][name])))

## Phase 6 handoff

Use `uq_pilot_analysis_filtered.laz`, `dtm_1m.tif` and `height_above_ground.tif` for building modelling. Keep `uq_pilot_cropped_full.laz` as the audit fallback for missing or suspicious returns. Exact paths and limitations are documented in `PHASE5_TERRAIN_LIDAR_PREPARATION.md` and the machine-readable summary.